[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/TianshuangQiu/TorchCode/blob/master/templates/58_capsule_poly_overlap.ipynb)

# 🔴 Hard: Capsule-Convex Polygon Overlap

A **capsule** is the set of all points within distance `r` of a line segment (its spine). Given a capsule and a convex polygon, return `True` if they overlap.

This problem composes three primitives you have seen before. Identify the three cases, state the reduction for each, then implement.

### Signature
```python
def capsule_poly_overlap(capsule: np.ndarray, poly: np.ndarray) -> bool:
    # capsule: (5,) float — [x1, y1, x2, y2, r]
    # poly:    (V, 2) float — convex polygon vertices, CCW order
    # returns: bool
```

### Rules
- Do **NOT** use Python `for` loops over polygon vertices (vectorise over V)
- Touching (distance exactly `r`) counts as overlapping

### Example
```
capsule = [0, 1, 4, 1, 0.5]         # horizontal spine y=1, radius 0.5
square  = [[1,0],[3,0],[3,2],[1,2]]  # 2x2 square
output: True  (spine passes through the square)

capsule2 = [0, 5, 4, 5, 0.3]        # far above, r too small
output: False
```

> **Reduction step (say this before coding):** `overlap` is True iff the minimum distance from the capsule spine to the polygon *region* is less than or equal to `r` — which equals 0 when any spine endpoint is inside the polygon or the spine crosses any polygon edge, and otherwise equals the minimum segment-to-segment distance from the spine to each of the V polygon edges.

In [ ]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass

In [ ]:
import numpy as np

In [ ]:
# ✏️ YOUR IMPLEMENTATION HERE

def capsule_poly_overlap(capsule, poly):
    # capsule: (5,) — [x1, y1, x2, y2, r]
    # poly:    (V, 2) — convex polygon, CCW
    # returns: bool
    #
    # Hint: three cases
    # 1. Is either endpoint of the spine inside the polygon?
    #    For CCW convex polygon: cross(edge_i, pt-v_i) >= 0 for ALL i
    # 2. Does the spine segment cross any polygon edge?
    #    Cross-product orientation test (see problem #49)
    # 3. Is the min distance from spine to any polygon edge <= r?
    #    Parametric segment-to-segment distance (see problem #56)
    pass  # Replace this

In [ ]:
# 🧪 Test your implementation
square = np.array([[1.0,0.0],[3.0,0.0],[3.0,2.0],[1.0,2.0]])
cap1 = np.array([0.0, 1.0, 4.0, 1.0, 0.5])  # spine crosses square
cap2 = np.array([0.0, 5.0, 4.0, 5.0, 0.3])  # far above
print("crosses:", capsule_poly_overlap(cap1, square))  # expect True
print("far away:", capsule_poly_overlap(cap2, square)) # expect False

In [ ]:
# ✅ Inline test suite
import numpy as np, time

# ── Test 1: spine entirely inside polygon ─────────────────────────────────
sq = np.array([[0.0,0.0],[4.0,0.0],[4.0,4.0],[0.0,4.0]])
cap1 = np.array([1.0, 1.0, 3.0, 3.0, 0.2])
assert capsule_poly_overlap(cap1, sq) == True, "Spine inside polygon → overlap"
print("Test 1 passed: spine inside polygon")

# ── Test 2: spine crosses polygon boundary ────────────────────────────────
sq2 = np.array([[0.0,0.0],[2.0,0.0],[2.0,2.0],[0.0,2.0]])
cap2 = np.array([-1.0, 1.0, 3.0, 1.0, 0.1])
assert capsule_poly_overlap(cap2, sq2) == True, "Spine crossing boundary → overlap"
print("Test 2 passed: spine crosses boundary")

# ── Test 3: capsule fully separated, r too small ──────────────────────────
cap3 = np.array([4.0, 0.5, 6.0, 0.5, 0.5])
assert capsule_poly_overlap(cap3, sq2) == False, "Separated capsule → no overlap"
print("Test 3 passed: fully separated")

# ── Test 4: radius just reaches a polygon vertex ──────────────────────────
tri = np.array([[0.0,0.0],[2.0,0.0],[1.0,2.0]])
cap_reach = np.array([0.5, -1.0, 1.5, -1.0, 1.2])
cap_miss  = np.array([0.5, -1.0, 1.5, -1.0, 0.9])
assert capsule_poly_overlap(cap_reach, tri) == True,  "r reaches vertex → overlap"
assert capsule_poly_overlap(cap_miss,  tri) == False, "r too small → no overlap"
print("Test 4 passed: radius near vertex")

# ── Test 5: 50-pair reference comparison + timing ─────────────────────────
def _inside_convex(pt, poly):
    edges = np.roll(poly, -1, axis=0) - poly
    vecs  = pt - poly
    cross = edges[:,0]*vecs[:,1] - edges[:,1]*vecs[:,0]
    return bool((cross >= 0).all())

def _seg_dist(p1, p2, p3, p4):
    EPS = 1e-12
    d1 = p2-p1; d2 = p4-p3; rv = p1-p3
    a = d1@d1; e = d2@d2; b = d1@d2; c = d1@rv; f = d2@rv
    denom = a*e - b*b
    s = float(np.clip((b*f-c*e)/denom,0,1)) if denom>EPS else 0.0
    t = float(np.clip((b*s+f)/e,0,1)) if e>EPS else 0.0
    s = float(np.clip((b*t-c)/a,0,1)) if a>EPS else 0.0
    return float(np.linalg.norm(p1+s*d1-p3-t*d2))

def reference(capsule, poly):
    p1, p2, r = capsule[:2], capsule[2:4], capsule[4]
    if _inside_convex(p1, poly) or _inside_convex(p2, poly):
        return True
    V = len(poly)
    for i in range(V):
        if _seg_dist(p1, p2, poly[i], poly[(i+1)%V]) <= r:
            return True
    return False

rng = np.random.default_rng(7)
def rand_square(rng):
    x, y = rng.uniform(0, 8), rng.uniform(0, 8)
    s = rng.uniform(1, 3)
    return np.array([[x,y],[x+s,y],[x+s,y+s],[x,y+s]])

t0 = time.time()
for _ in range(50):
    poly = rand_square(rng)
    pts = rng.uniform(0, 10, (2, 2))
    r = float(rng.uniform(0.2, 2.0))
    cap = np.array([pts[0,0], pts[0,1], pts[1,0], pts[1,1], r])
    expected = reference(cap, poly)
    got = capsule_poly_overlap(cap, poly)
    assert got == expected, f"Mismatch: expected {expected}, got {got}, cap={cap}, poly={poly}"
elapsed = time.time() - t0
assert elapsed < 3.0, f"Too slow: {elapsed:.2f}s"
print(f"Test 5 passed: 50 random pairs vs reference ({elapsed:.3f}s)")

print("\nAll tests passed!")